# Predicting Steam Review Recommendations with Logistic Regression

This project tests whether the text of a Steam review can predict whether the reviewer recommends the game. This is a binary classification problem because the target has two classes: recommended and not recommended.


## 1. Import Libraries

TF-IDF converts review text into numeric features, and logistic regression is used to classify the reviews.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report


## 2. Load and Clean the Data

This notebook uses `steam_game_reviews_730945.csv`, which contains 730,945 Steam reviews. The code keeps only the review text and recommendation label, removes missing values, and samples at most 20,000 reviews to keep runtime manageable.


In [ ]:
df = pd.read_csv("steam_game_reviews_730945.csv")

review_col = "review" if "review" in df.columns else "review_text"
label_col = "voted_up" if "voted_up" in df.columns else "recommended"

df = df[[review_col, label_col]].dropna()
df.columns = ["review", "recommended"]

if df["recommended"].dtype == "object":
    df["recommended"] = df["recommended"].astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False,
        "yes": True, "no": False
    })

df = df.dropna()
df = df[df["review"].astype(str).str.strip() != ""]

if len(df) > 20000:
    df = df.sample(20000, random_state=42)

df.head()


## 3. Class Balance

This chart shows how many reviews belong to each class. A large imbalance can make accuracy less informative.


In [ ]:
counts = df["recommended"].value_counts().sort_index()
counts.index = ["Not Recommended" if x == False else "Recommended" for x in counts.index]

counts.plot(kind="bar")
plt.title("Steam Review Class Balance")
plt.xlabel("Review Label")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()


## 4. Train/Test Split

The dataset is split into 80% training data and 20% testing data. Stratification keeps the class proportions similar in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["recommended"],
    test_size=0.2,
    random_state=42,
    stratify=df["recommended"]
)


## 5. Convert Text into Features

TF-IDF gives higher weight to words that are useful for distinguishing reviews while reducing the importance of very common words.


In [ ]:
vec = TfidfVectorizer(max_features=5000, stop_words="english", min_df=2)

X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)


## 6. Train the Model

Logistic regression estimates the probability that a review belongs to the recommended class and converts that probability into a class prediction.


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

pred = model.predict(X_test_vec)
prob = model.predict_proba(X_test_vec)[:, 1]


## 7. Evaluate Model Performance

Accuracy measures overall correctness. Precision measures how often predicted recommendations were correct, recall measures how many actual recommendations were found, and F1 balances precision and recall.


In [ ]:
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})

metrics


In [ ]:
metrics.set_index("Metric")["Score"].plot(kind="bar")
plt.title("Logistic Regression Performance")
plt.xlabel("")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()


## 8. Confusion Matrix

The confusion matrix separates correct and incorrect predictions for recommended and not-recommended reviews.


In [ ]:
cm = confusion_matrix(y_test, pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Recommended", "Recommended"]
)

disp.plot()
plt.title("Confusion Matrix")
plt.show()

print(classification_report(
    y_test,
    pred,
    target_names=["Not Recommended", "Recommended"]
))


## 9. Most Influential Words

Positive coefficients are words associated with recommended reviews. Negative coefficients are words associated with not-recommended reviews.


In [ ]:
features = vec.get_feature_names_out()
coef = model.coef_[0]

words = pd.DataFrame({
    "word": features,
    "coefficient": coef
})

positive = words.nlargest(10, "coefficient").sort_values("coefficient")
negative = words.nsmallest(10, "coefficient").sort_values("coefficient")

positive.plot(x="word", y="coefficient", kind="barh", legend=False)
plt.title("Words Most Associated with Recommended Reviews")
plt.xlabel("Coefficient")
plt.ylabel("")
plt.show()


In [ ]:
negative.plot(x="word", y="coefficient", kind="barh", legend=False)
plt.title("Words Most Associated with Not-Recommended Reviews")
plt.xlabel("Coefficient")
plt.ylabel("")
plt.show()


## 10. Five Reviews the Model Got Wrong

These five incorrect predictions can be examined for sarcasm, mixed opinions, short reviews, unusual wording, or game-specific language that the model may not understand well.


In [ ]:
results = pd.DataFrame({
    "review": X_test,
    "actual": y_test,
    "predicted": pred,
    "recommended_probability": prob
})

wrong = results[results["actual"] != results["predicted"]].copy()

wrong.head(5)


## 11. Interpretation

Use the model scores, confusion matrix, influential words, and five incorrect reviews to explain how well the model answers the question of whether Steam review text can predict a recommendation.


## 12. Limitations

The model only uses review text and ignores features such as playtime, game genre, price, updates, and review date. TF-IDF also does not fully understand context or sarcasm. Steam reviewers may not represent all players.

If AI assistance is used, the dataset columns, outputs, plots, and interpretations should be checked against the actual data before including them in the final post.
